# F-03: Partial Pooling across all towers (RFm)

Follow-up to F-02. Compares three ways to share data across towers, all using **stocking-density** livestock (D-29):

- **solo** — one model per tower (per-tower training only).
- **full pool** — one model on Towers 2+4+9 stacked, *no* tower identity (F-02 style).
- **partial pool** — pooled + **tower indicator** dummies (`is_t2/is_t4/is_t9`): shared relationships but tower-specific level/intercept.

Evaluated on **all three towers**: Towers 4 & 9 on the standard test (2022-23); **Tower 2 on its D-15 custom split** (train 2018 / test Jan-May 2019) — the seasonal-mismatch caveat (D-15/D-19) still applies, so T2 numbers gauge whether pooling helps, not absolute skill.

BASE generic features = driver_m (met) + gap-filled FCO2 + AUX + livestock **density** + grazing flag.

In [1]:
from pathlib import Path
import datetime
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

HOURLY=Path("../../data/Hourly"); RESULTS=Path("../../results")
PLAUS_LOW,PLAUS_HIGH=-500,3000
N_REPS,MASK_FRAC=5,0.25
SCENARIOS={"vs":1,"s":4,"m":32,"l":288,"m1":"mixed"}
LSU={"cattle":1.0,"sheep":0.1,"lamb":0.05}
AUX=["_hour_sin","_hour_cos","_doy_sin","_doy_cos"]
CATCHMENT_AREA_HA={1:4.81,2:6.65,3:6.62,4:7.75,5:6.47,6:3.86,7:2.60,8:7.02,9:7.75,10:1.82,11:1.76,12:1.78,13:1.75,14:1.72,15:1.54}
TOWERS=[2,4,9]

## 1  Configs

In [2]:
C4="Catchment 4 After  2013/08/13"
def cfg(t,cat,catstr):
    return {"tower":t,"area_ha":CATCHMENT_AREA_HA[cat],
        "target":f"FCH4_1_1_1 [Tower {t}]","ssitc":f"FCH4_SSITC_TEST_1_1_1 [Tower {t}]",
        "sw":f"SWIN_1_1_1 [Tower {t}]","ta":f"TA_0_0_1 [Tower {t}]","vpd":f"VPD_0_0_1 [Tower {t}]",
        "ppfd":f"PPFD_1_1_1 [Tower {t}]","ustar":f"USTAR_0_0_1 [Tower {t}]","ws":f"WS_0_0_1 [Tower {t}]",
        "rn":f"RN_1_1_1 [Tower {t}]","precip":f"Precipitation (mm) [{catstr}]","ts":"TS_1_1_1 [Tower 9]",
        "swc":f"Soil Moisture @ 10cm Depth (%) [{catstr}]","shf":f"SHF_1_1_1 [Tower {t}]","fc":f"FC_1_1_1 [Tower {t}]",
        "livestock":{sp:f"{sp}_{catstr}" for sp in LSU},
        "train_yrs":[2018] if t==2 else list(range(2018,2022)),
        "test_yrs":[2019] if t==2 else list(range(2022,2024))}
TOWER_CONFIGS={2:cfg(2,2,"Catchment 2"),4:cfg(4,4,C4),9:cfg(9,9,"Catchment 9")}

## 2  Load

In [3]:
df=pd.read_csv(HOURLY/"consolidated_hourly.csv",low_memory=False)
df["Datetime"]=pd.to_datetime(df["Datetime"],format="mixed"); df=df.set_index("Datetime")
fco2=pd.read_csv(HOURLY/"fco2_gapfilled.csv",low_memory=False)
fco2["Datetime"]=pd.to_datetime(fco2["Datetime"],format="mixed"); fco2=fco2.set_index("Datetime")
for _t in TOWERS:
    c=f"FC_1_1_1 [Tower {_t}]"
    if c in df.columns: df[c]=fco2[f"FC_gapfilled [Tower {_t}]"]
df_raw=df; print("loaded",df_raw.shape)

loaded (70153, 449)


## 3  Functions

In [4]:
def insert_calendar_gaps(d,target,test_yrs,gh,n_reps=N_REPS,seed=0):
    tm=d.index.year.isin(test_yrs); tt=d.index[tm]; va=d.loc[tm,target].notna().values
    n=len(tt); tn=max(1,int(va.sum()*MASK_FRAC)); rb=np.random.default_rng(seed); reps=[]
    for _ in range(n_reps):
        rng=np.random.default_rng(int(rb.integers(0,2**31))); occ=np.zeros(n,bool); m=0
        for sp in rng.permutation(n):
            if m>=tn: break
            g=int(rng.choice([1,4,32,288])) if gh=="mixed" else gh; ep=min(int(sp)+g,n)
            if occ[sp:ep].any(): continue
            occ[sp:ep]=True; m+=int(va[sp:ep].sum())
        reps.append(tt[occ & va])
    return reps

def generic_frame(t):
    c=TOWER_CONFIGS[t]; d=df_raw.copy(); tgt=c["target"]
    d.loc[~d[c["ssitc"]].isin([0,1]),tgt]=np.nan
    d.loc[d[tgt].notna() & ~d[tgt].between(PLAUS_LOW,PLAUS_HIGH,inclusive="both"),tgt]=np.nan
    h,doy=d.index.hour,d.index.dayofyear
    d["_hour_sin"]=np.sin(2*np.pi*h/24); d["_hour_cos"]=np.cos(2*np.pi*h/24)
    d["_doy_sin"]=np.sin(2*np.pi*doy/365); d["_doy_cos"]=np.cos(2*np.pi*doy/365)
    liv=c["livestock"]; area=c["area_ha"]
    lsu=sum(d[liv[s]].fillna(0)*w for s,w in LSU.items())
    g=pd.DataFrame(index=d.index); g["target"]=d[tgt]
    for k in ["sw","ta","vpd","ppfd","ustar","ws","rn","precip","ts","swc","shf","fc"]: g[k]=d[c[k]]
    for a in AUX: g[a]=d[a]
    g["lsu_dens"]=lsu/area
    g["graze"]=(sum(d[liv[s]].fillna(0) for s in LSU)>0).astype(float)
    for tt in TOWERS: g[f"is_t{tt}"]=1.0 if tt==t else 0.0
    g["_year"]=d.index.year
    return g, tgt

BASE_FEAT=["sw","ta","vpd","ppfd","ustar","ws","rn","precip","ts","swc","shf","fc"]+AUX+["lsu_dens","graze"]
DUMMIES=[f"is_t{t}" for t in TOWERS]
PARTIAL_FEAT=BASE_FEAT+DUMMIES

In [5]:
def fit(feat, train_df):
    imp=SimpleImputer(strategy="mean")
    rf=RandomForestRegressor(n_estimators=500,min_samples_leaf=5,n_jobs=-1,random_state=42)
    rf.fit(imp.fit_transform(train_df[feat].values), train_df["target"].values)
    return rf,imp

def eval_on(rf,imp,feat,gframe,c):
    recs=[]
    rep_gaps={sc:insert_calendar_gaps(gframe,"target",c["test_yrs"],gh) for sc,gh in SCENARIOS.items()}
    for sc,reps in rep_gaps.items():
        for rep,ts in enumerate(reps):
            if len(ts)<5: continue
            yt=gframe.loc[ts,"target"].values; yp=rf.predict(imp.transform(gframe.loc[ts,feat].values))
            recs.append({"scenario":sc,"rep":rep,"R2":r2_score(yt,yp),
                "RMSE":float(mean_squared_error(yt,yp)**0.5),"MBE":float(np.mean(yp-yt)),
                "MAE":float(mean_absolute_error(yt,yp)),"n_test":len(yt)})
    return pd.DataFrame(recs)

## 4  Build frames, train 3 model variants

In [6]:
frames={t:generic_frame(t)[0] for t in TOWERS}
# pooled training rows (all towers, each its own train years), dropna on BASE+target
def train_rows(feat):
    parts=[]
    for t in TOWERS:
        g=frames[t]; yrs=TOWER_CONFIGS[t]["train_yrs"]
        parts.append(g[g["_year"].isin(yrs)][feat+["target"]].dropna())
    return pd.concat(parts,ignore_index=True)

pool_train=train_rows(PARTIAL_FEAT)
print("pooled train rows:",len(pool_train),"| per tower:",
      {t:int((frames[t]["_year"].isin(TOWER_CONFIGS[t]["train_yrs"]) & frames[t]["target"].notna()).sum()) for t in TOWERS})

rf_full,imp_full = fit(BASE_FEAT, pool_train)              # full pool (no tower id)
rf_part,imp_part = fit(PARTIAL_FEAT, pool_train)           # partial pool (+ dummies)
solo={}
for t in TOWERS:
    g=frames[t]; tr=g[g["_year"].isin(TOWER_CONFIGS[t]["train_yrs"])][BASE_FEAT+["target"]].dropna()
    solo[t]=fit(BASE_FEAT, tr)
print("trained: full, partial, solo x3")

pooled train rows: 12558 | per tower: {2: 3629, 4: 12033, 9: 5387}


trained: full, partial, solo x3


## 5  Evaluate all three towers

In [7]:
results=[]
for t in TOWERS:
    g=frames[t]; c=TOWER_CONFIGS[t]
    for name,(rf,imp,feat) in {
        "solo":(solo[t][0],solo[t][1],BASE_FEAT),
        "full_pool":(rf_full,imp_full,BASE_FEAT),
        "partial_pool":(rf_part,imp_part,PARTIAL_FEAT),
    }.items():
        m=eval_on(rf,imp,feat,g,c); m["tower"]=f"Tower {t}"; m["variant"]=name
        results.append(m)
res=pd.concat(results,ignore_index=True)
print("done; rows:",len(res))

done; rows: 225


## 6  Results — median R2 by tower x variant x scenario

In [8]:
VARS=["solo","full_pool","partial_pool"]
for t in TOWERS:
    sub=res[res.tower==f"Tower {t}"]
    tbl=sub.groupby(["variant","scenario"])["R2"].median().unstack("scenario").reindex(index=VARS).reindex(columns=list(SCENARIOS)).round(3)
    print(f"\n=== Tower {t} ===")
    print(tbl.to_string())
print("\n--- Overall median R2 by tower x variant (across scenarios) ---")
print(res.groupby(["tower","variant"])["R2"].median().unstack("variant").reindex(columns=VARS).round(3).to_string())


=== Tower 2 ===
scenario         vs      s      m      l     m1
variant                                        
solo         -0.712 -0.437 -0.426 -0.474 -0.141
full_pool    -0.301 -0.147 -0.259 -0.243 -0.238
partial_pool -0.245 -0.125 -0.216 -0.201 -0.201

=== Tower 4 ===
scenario         vs      s      m      l     m1
variant                                        
solo          0.248  0.147  0.180  0.007 -0.086
full_pool     0.239  0.151  0.236  0.026 -0.110
partial_pool  0.238  0.153  0.236  0.028 -0.107

=== Tower 9 ===
scenario         vs      s      m      l     m1
variant                                        
solo         -0.054 -0.046 -0.010 -0.145 -0.008
full_pool     0.287  0.217  0.294  0.207  0.250
partial_pool  0.293  0.220  0.292  0.204  0.251

--- Overall median R2 by tower x variant (across scenarios) ---
variant   solo  full_pool  partial_pool
tower                                  
Tower 2 -0.426     -0.230        -0.179
Tower 4  0.098      0.065         0.067
Towe

## 7  Append to benchmarks.csv

In [9]:
bench=RESULTS/"benchmarks.csv"; today=datetime.date.today().isoformat()
ex=pd.read_csv(bench); ex=ex[ex["replication"]!="F-03"]
rows=[]
for r in res.to_dict("records"):
    rows.append({"replication":"F-03","model":f"RFm_{r['variant']}","tower":r["tower"],
        "feature_set":"density+dummies" if r["variant"]=="partial_pool" else "density",
        "scenario":r["scenario"],"rep":r["rep"],"split":"pool_T2+4+9",
        "R2":round(r["R2"],4),"RMSE":round(r["RMSE"],4),"MAE":round(r["MAE"],4),"MBE":round(r["MBE"],4),
        "n_test":r["n_test"],"date":today,"notes":"F03 partial pooling (tower dummies) vs full/solo; stocking density (D-30)"})
new=pd.DataFrame(rows); comb=pd.concat([ex,new],ignore_index=True); comb.to_csv(bench,index=False)
print(f"Wrote {len(new)} F-03 rows. Total {len(comb)}.")

Wrote 225 F-03 rows. Total 2065.
